# Phase 1: Data Understanding & Schema Inspection

**Milestone**: Milestone 1.5 (Raw Data Ingestion Module)  
**Dataset**: CPCB Surface Water Quality Monitoring Dataset (`data/interim/cpcb_labeled.csv`)  
**Objective**: Ingest, validate, and explore the labeled CPCB dataset using the centralized `src.data.load_data` module.

In [1]:
# Cell 1: Environment Setup & Module Imports
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.load_data import load_raw_data, validate_schema, load_config

print("Loaded src.data.load_data successfully.")

Loaded src.data.load_data successfully.


In [2]:
# Cell 2: Central Data Ingestion & Frozen Schema Validation
config = load_config()
df = load_raw_data(config=config, validate=True)

print(f"\n[Schema Validation Success]")
print(f"  * Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"  * Target Column: {config['target_column']}")
print(f"  * Candidate Features: {config['feature_columns']}")

2026-10-05 15:52:38,229 [INFO] load_data: Loading labeled CPCB dataset from: D:\Coding\College\Data science\CP\data\interim\cpcb_labeled.csv
2026-10-05 15:52:38,272 [INFO] load_data: ================ DATASET SCHEMA VALIDATION AUDIT ================
2026-10-05 15:52:38,274 [INFO] load_data:   Dataset Shape: 3287 rows x 70 columns
2026-10-05 15:52:38,274 [INFO] load_data:   Duplicate Rows: 0
2026-10-05 15:52:38,275 [INFO] load_data:   Target Column ('wqi_class'): Present (Dtype: object)
2026-10-05 15:52:38,275 [INFO] load_data:   Secondary Target ('Potability'): Present (Dtype: int64)
2026-10-05 15:52:38,275 [INFO] load_data:   Candidate Feature Columns Missingness Audit:
2026-10-05 15:52:38,275 [INFO] load_data:     * Potential of Hydrogen (pH)          :    0 missing ( 0.00%)
2026-10-05 15:52:38,278 [INFO] load_data:     * Dissolved oxygen (mg/L)             :   90 missing ( 2.74%)
2026-10-05 15:52:38,278 [INFO] load_data:     * Biochemical Oxygen Demand (mg/L)    :  178 missing ( 5.42


[Schema Validation Success]
  * Dataset Shape: 3287 rows x 70 columns
  * Target Column: wqi_class
  * Candidate Features: ['Potential of Hydrogen (pH)', 'Dissolved oxygen (mg/L)', 'Biochemical Oxygen Demand (mg/L)', 'Fecal Coliform (MPN/100mL)']


## 1. Ground-Truth Target Distributions

Inspect the distributions for the primary multi-class target (`wqi_class`) and the secondary binary target (`Potability`).

In [3]:
# Multi-class target breakdown
print("=== Primary Target (wqi_class) Breakdown ===")
class_counts = df["wqi_class"].value_counts()
class_pcts = df["wqi_class"].value_counts(normalize=True) * 100
target_dist = pd.DataFrame({"Count": class_counts, "Percentage (%)": class_pcts.round(2)})
display(target_dist)

# Binary potability breakdown
print("\n=== Secondary Target (Potability) Breakdown ===")
pot_counts = df["Potability"].value_counts()
pot_pcts = df["Potability"].value_counts(normalize=True) * 100
pot_dist = pd.DataFrame({"Count": pot_counts, "Percentage (%)": pot_pcts.round(2)})
pot_dist.index = ["Non-Potable (0)", "Potable (1)"]
display(pot_dist)

=== Primary Target (wqi_class) Breakdown ===


,Count,Percentage (%)
wqi_class,,
Unsuitable,1796,54.64
Very Poor,919,27.96
Poor,455,13.84
Good,110,3.35
Excellent,7,0.21



=== Secondary Target (Potability) Breakdown ===


,Count,Percentage (%)
Non-Potable (0),3170,96.44
Potable (1),117,3.56


## 2. Frozen Candidate Features Audit

Inspect descriptive statistics and missingness for the four candidate water quality parameters.

In [4]:
features = config["feature_columns"]
stats_df = df[features].describe().T
stats_df["missing_count"] = df[features].isna().sum()
stats_df["missing_pct"] = (df[features].isna().mean() * 100).round(2)
display(stats_df[["count", "mean", "std", "min", "50%", "max", "missing_count", "missing_pct"]])

,count,mean,std,min,50%,max,missing_count,missing_pct
Potential of Hydrogen (pH),3287.0,7.777417,0.445559,3.45,7.8,9.2,0,0.00
Dissolved oxygen (mg/L),3197.0,6.235846,2.034560,0.30,6.4,28.0,90,2.74
Biochemical Oxygen Demand (mg/L),3109.0,7.471759,10.468671,1.10,3.9,127.0,178,5.42
Fecal Coliform (MPN/100mL),3020.0,36640.541159,502913.179549,2.00,110.0,22000000.0,267,8.12


## 3. Geographic & Administrative Scope

Analyze monitoring stations and state representation across Maharashtra and Uttar Pradesh.

In [5]:
print(f"Total Unique Stations: {df['Station'].nunique()}")
print(f"Total Unique Districts: {df['District'].nunique()}")
print("\nState Representation:")
display(df.groupby("State").agg(Total_Observations=("Station", "count"), Unique_Stations=("Station", "nunique")))

Total Unique Stations: 337
Total Unique Districts: 71

State Representation:


,Total_Observations,Unique_Stations
State,,
Maharashtra,1896,209
Uttar Pradesh,1391,128


## 4. Target Leakage Verification

Confirm that the continuous mathematical index `wqi_score` is strictly excluded from `feature_columns`.

In [6]:
assert "wqi_score" not in config["feature_columns"], "CRITICAL: Target leakage in feature_columns!"
print("Target Leakage Verification: PASSED (wqi_score is excluded from features)")
print("\nData Understanding Notebook complete. Ready for Phase 2 Preprocessing & Feature Engineering.")

Target Leakage Verification: PASSED (wqi_score is excluded from features)

Data Understanding Notebook complete. Ready for Phase 2 Preprocessing & Feature Engineering.
